In [ ]:
import pandas as pd
import numpy as np

In [ ]:
matches = pd.read_csv("/Users/ishagarg/Documents/PythonProjects/ipl-player-intelligence/data/matches.csv")
deliveries = pd.read_csv("/Users/ishagarg/Documents/PythonProjects/ipl-player-intelligence/data/deliveries.csv")

In [ ]:
print("Matches shape:", matches.shape)
print("Deliveries shape:", deliveries.shape)

In [ ]:
# See all column names
print("MATCHES columns:", matches.columns.tolist())
print("\nDELIVERIES columns:", deliveries.columns.tolist())

In [ ]:
print("Matches missing values:")
print(matches.isnull().sum())

print("\nDeliveries missing values:")
print(deliveries.isnull().sum())

In [ ]:
matches.head()
deliveries.head()

In [ ]:
matches["season"].unique()


In [ ]:
matches["season"] = matches["season"].replace({'2007/08':'2008','2009/10':'2010','2020/21':'2020'}, inplace = True)

In [ ]:
print("Total numbers of players who bowled: ",
deliveries["bowler"].nunique())

In [ ]:
print("Total numbers of players who batted: ",
deliveries["batter"].nunique())

In [ ]:
top_scorers = (deliveries
    .groupby('batter')['batsman_runs']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index())

top_scorers

In [ ]:
top_scorers.columns = ['Player', 'Total Runs']
print(top_scorers)

In [ ]:
top_bowlers = deliveries.groupby('bowler')['is_wicket'].sum().sort_values(ascending = False).head(10).reset_index()

In [ ]:
top_bowlers

In [ ]:
balls_faced = deliveries.groupby('batter').size()
print(balls_faced.head(10))

runs_scored = deliveries.groupby('batter')['batsman_runs'].sum()
print("\n",runs_scored.head(10))

strike_rate = (runs_scored / balls_faced * 100).round(2)
# strike_rate.shape
# 673
qualified = strike_rate[balls_faced >= 500].sort_values(ascending=False).head(10)

qualified

In [ ]:
bowler_runs = deliveries.groupby('bowler')['total_runs'].sum()
bowler_balls = deliveries.groupby('bowler').size()
bowler_overs = bowler_balls / 6

economy = (bowler_runs / bowler_overs).round(2)
qualified_bowlers = economy[bowler_overs >= 200].sort_values().head(10)

print(qualified_bowlers)

In [ ]:
# Create a player summary table — to load this into Supabase
player_summary = pd.DataFrame({
    'player': runs_scored.index,
    'total_runs': runs_scored.values,
    'balls_faced': balls_faced.reindex(runs_scored.index).values,
    'strike_rate': strike_rate.reindex(runs_scored.index).round(2).values
}).dropna().sort_values('total_runs', ascending=False)

player_summary.to_csv('/Users/ishagarg/Documents/PythonProjects/ipl-player-intelligence/data/player_summary.csv', index=False)
print("Saved! Rows:", len(player_summary))

In [ ]:
##### Supabase Upload

In [ ]:
import os
from supabase import create_client
from dotenv import load_dotenv
import pandas as pd

load_dotenv("/Users/ishagarg/Documents/PythonProjects/ipl-player-intelligence/.env")

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")

supabase = create_client(url, key)

# print(os.getcwd())
# oai_key = os.getenv("OPENAI_API_KEY")
# print(f"Key loaded: {oai_key}..." if oai_key else "❌ Key not found")

# # Print to verify credentials loaded
# print("URL:", os.getenv("SUPABASE_URL"))
# print("Key:", os.getenv("SUPABASE_KEY")[:8], "...")

# Now test query
result = supabase.table('player_summary').select("*").limit(1).execute()
print(result.data)

def query_supabase(table, filters=None):
    """
    Simple query function - name of the table to query
    filters: optional dict of column:value filters
    """
    query = supabase.table(table).select("*")
    
    if filters:
        for column, value in filters.items():
            query = query.eq(column, value)
    
    result = query.execute()
    return pd.DataFrame(result.data)

# Test it
df = query_supabase('matches')
print(f"✅ Got {len(df)} rows from matches table")
print(df.head())



In [ ]:
# Load your CSVs
matches = pd.read_csv('data/matches.csv')
deliveries = pd.read_csv('data/deliveries.csv')
player_summary = pd.read_csv('data/player_summary.csv')

# Clean column names — Supabase doesn't like spaces
matches.columns = matches.columns.str.lower().str.replace(' ', '_')
deliveries.columns = deliveries.columns.str.lower().str.replace(' ', '_')
matches["season"] = matches["season"].replace({'2007/08':'2008','2009/10':'2010','2020/21':'2020'}, inplace = True)

print("Ready to upload")
print(f"Matches: {len(matches)} rows")
print(f"Deliveries: {len(deliveries)} rows")
print(f"Player summary: {len(player_summary)} rows")

In [ ]:
matches["season"].unique()

In [ ]:
# Upload in batches — Supabase has row limits per request
def upload_in_batches(supabase, table_name, df, batch_size=500):
    # df = df.where(pd.notnull(df), None)  # replace NaN with None
    df = df.replace({np.nan: None})
    records = df.to_dict('records')
    total = len(records)
    
    for i in range(0, total, batch_size):
        batch = records[i:i+batch_size]
        supabase.table(table_name).insert(batch).execute()
        print(f"Uploaded {min(i+batch_size, total)}/{total} rows")
    
    print(f"✓ {table_name} upload complete")

# Upload matches first (smaller table)
upload_in_batches(supabase, 'matches', matches)

In [ ]:
upload_in_batches(supabase, 'player_summary', player_summary)

In [ ]:
# OpenAI API Calls

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "user", "content": "Say hello in 5 words"}
    ]
)
print(response.choices[0].message.content)

In [ ]:
def ask_ipl(schema, question):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": schema},
            {"role": "user", "content": question}
        ]
    )
    
    return response.choices[0].message.content

# Test it
# Tell the AI about your database schema
schema = """
You have access to these tables:
    
player_summary: player(name), total_runs, balls_faced, strike_rate
matches: id, season, city, date, team1, team2, winner, player_of_match
    
Write a PostgreSQL SQL query to answer the user's question.
Return ONLY the SQL query, nothing else.
"""
question = "Who are the top 5 players by total runs?"
sql = ask_ipl(schema, question)
print("Generated SQL:")
print(sql)


In [ ]:
# Get top 20 players from your database
result = supabase.table('player_summary')\
    .select("*")\
    .order('total_runs', desc=True)\
    .limit(20)\
    .execute()

players_df = pd.DataFrame(result.data)
print(players_df)

In [ ]:
# Convert player data to readable text for the AI
players_text = players_df.to_string(index=False)

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {
            "role": "system",
            "content": """You are an expert IPL cricket analyst with deep knowledge 
            of player performance, batting techniques, and match situations. 
            Give specific, insight-driven analysis — not generic commentary."""
        },
        {
            "role": "user",
            "content": f"""Here is IPL player performance data:

{players_text}

Question: Who are the top 3 most valuable batsmen and why? 
Consider both total runs and strike rate in your analysis."""
        }
    ]
)

print(response.choices[0].message.content)

In [ ]:
def generate_sql(question):
    schema = """
    You have access to these PostgreSQL tables:
    
    player_summary:
    - player (TEXT): player name
    - total_runs (FLOAT): total runs scored in IPL career
    - balls_faced (FLOAT): total balls faced
    - strike_rate (FLOAT): career strike rate
    
    matches:
    - id (INTEGER): match id
    - season (INTEGER): IPL season year
    - city (TEXT): city where match was played
    - date (TEXT): match date
    - team1 (TEXT): first team
    - team2 (TEXT): second team
    - winner (TEXT): winning team
    - player_of_match (TEXT): player of the match
    - toss_winner (TEXT): toss winning team
    - toss_decision (TEXT): bat or field
    - result (TEXT): match result type
    - result_margin (FLOAT): margin of victory
    - venue (TEXT): stadium name
    
    Rules:
    - Return ONLY the SQL query, no explanation, no markdown
    - Use lowercase column names
    - Always use LIMIT 20 unless asked for specific number
    - For player names use ILIKE for case-insensitive matching
    """
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": schema},
            {"role": "user", "content": question}
        ]
    )
    
    return response.choices[0].message.content.strip()

# Test it
question = "Who are the top 5 players by strike rate with more than 1000 balls faced?"
sql = generate_sql(question)
print("Generated SQL:")
print(sql)

In [ ]:
def run_query(question):
    # Step 1: generate SQL from question
    sql = generate_sql(question)
    print(f"Generated SQL:\n{sql}\n")
    
    # Step 2: clean any markdown the AI might add
    import re
    sql = re.sub(r'```sql|```', '', sql).strip()
    
    # Step 3: run against Supabase using postgrest syntax
    # For now we'll use pandas as our query engine against local data
    # Full SQL execution via Supabase comes in Day 4
    print("Running query...")
    
    # Step 4: generate AI insight from results
    result = supabase.table('player_summary')\
        .select("*")\
        .order('strike_rate', desc=True)\
        .limit(5)\
        .execute()
    
    df = pd.DataFrame(result.data)
    
    # Step 5: AI summarizes the result
    summary_response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are an IPL analyst. Summarize data insights in 3 sentences max. Be specific with numbers."},
            {"role": "user", "content": f"Question: {question}\n\nData:\n{df.to_string(index=False)}\n\nProvide a sharp insight."}
        ]
    )
    
    print("Result:")
    print(df)
    print("\nAI Insight:")
    print(summary_response.choices[0].message.content)

# Run it
run_query("Who are the top 5 players by strike rate with more than 1000 balls faced?")

In [ ]:
from openai import OpenAI
from supabase import create_client
import os
import pandas as pd

# Clients
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
supabase = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))

# Pull real data from Supabase
result = supabase.table('player_summary')\
    .select("*")\
    .order('total_runs', desc=True)\
    .limit(10)\
    .execute()

players_df = pd.DataFrame(result.data)

# Send to AI
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {
            "role": "system",
            "content": "You are an expert IPL cricket analyst. Give specific, insight-driven analysis."
        },
        {
            "role": "user",
            "content": f"""Here is IPL player performance data:

{players_df.to_string(index=False)}

Who are the top 3 most valuable batsmen and why? Consider both total runs and strike rate."""
        }
    ]
)

print(response.choices[0].message.content)